In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/contracts_2024.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(169, 6)
['Player (168)', 'Pos', 'Yrs', 'Value', 'AAV', 'Signing_Year']


,Player (168),Pos,Yrs,Value,AAV,Signing_Year
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Shohei Ohtani QO,SP/DH,10.0,700000000.0,70000000.0,2024.0
2,Yoshinobu Yamamoto,SP,12.0,325000000.0,27083333.0,2024.0
3,Aaron Nola QO,SP/SP2,7.0,172000000.0,24571429.0,2024.0
4,Jung Hoo Lee,OF,6.0,113000000.0,18833333.0,2024.0


In [2]:
df = df.dropna(how="all")

df = df.rename(columns={
    "Player (168)": "player_name",
    "Pos": "position",
    "Yrs": "years",
    "Value": "total_value",
    "AAV": "aav",
    "Signing_Year": "signing_year",
})

df["qualifying_offer"] = df["player_name"].str.endswith(" QO", na=False).astype(int)
df["player_name"] = df["player_name"].str.replace(" QO", "", regex=False).str.strip()

print(df.shape)
df.head()

(168, 7)


,player_name,position,years,total_value,aav,signing_year,qualifying_offer
1,Shohei Ohtani,SP/DH,10.0,700000000.0,70000000.0,2024.0,1
2,Yoshinobu Yamamoto,SP,12.0,325000000.0,27083333.0,2024.0,0
3,Aaron Nola,SP/SP2,7.0,172000000.0,24571429.0,2024.0,1
4,Jung Hoo Lee,OF,6.0,113000000.0,18833333.0,2024.0,0
5,Josh Hader,RP/CL,5.0,95000000.0,19000000.0,2024.0,1


In [3]:
print(df.isna().sum())
print()
print(df["position"].value_counts())
print()
print("Duplicate names:", df["player_name"].duplicated().sum())
print()
print(df.sort_values("total_value").head(10)[["player_name", "position", "years", "total_value"]])

player_name          0
position             0
years                0
total_value          1
aav                 15
signing_year         0
qualifying_offer     0
dtype: int64

position
RP             32
SP             19
C              17
SP/SP2          8
RP/SU8          8
SS              7
RP/CL           6
SP/SP3          6
RP/SU7          5
CF              5
SP/SP1          4
3B              4
1B              3
3B/1B           3
SP/SP4          3
RF              3
P               3
SP/SP5          2
RF/RF           2
RF/2B           2
CF/3B           2
LF              2
RF/RF/CF/CF     2
SP/DH           1
OF              1
CF/1B           1
DH/OF           1
OF/LF/DH        1
RP/RP           1
SS/3B/SS/OF     1
LF/DH           1
DH/RF           1
1B/DH           1
1B/OF           1
2B/RF/DH        1
SS/OF           1
C/C             1
3B/LF           1
RP/SP           1
2B/CF           1
1B/LF           1
3B/RF           1
1B/3B/SS        1
Name: count, dtype: int64

Duplicate names

In [4]:
before = len(df)

# 1. Drop rows with no real contract
df = df[(df["total_value"] > 0) & (df["years"] > 0)].copy()
print("Rows removed:", before - len(df))

# 2. Use the first listed position (SP/SP2 -> SP, RF/2B -> RF)
df["position"] = df["position"].str.split("/").str[0]

# 3. Hitter or pitcher
df["player_type"] = df["position"].apply(
    lambda p: "pitcher" if p in ["SP", "RP", "P"] else "hitter"
)

# 4. Recalculate AAV ourselves instead of trusting the spreadsheet
df["aav"] = df["total_value"] / df["years"]

# Checks
print(df.shape)
print(df["position"].value_counts())
print(df["player_type"].value_counts())
print("Duplicate names:", df["player_name"].duplicated().sum())
print(df.isna().sum())

Rows removed: 15
(153, 8)
position
RP    50
SP    43
C     13
SS     9
CF     8
3B     8
1B     6
RF     6
LF     3
OF     2
DH     2
2B     2
P      1
Name: count, dtype: int64
player_type
pitcher    94
hitter     59
Name: count, dtype: int64
Duplicate names: 5
player_name         0
position            0
years               0
total_value         0
aav                 0
signing_year        0
qualifying_offer    0
player_type         0
dtype: int64


In [5]:
dupes = df[df["player_name"].duplicated(keep=False)].sort_values("player_name")
dupes[["player_name", "position", "years", "total_value", "aav"]]

,player_name,position,years,total_value,aav
53,Hector Neris,RP,1.0,9000000.0,9000000.0
148,Hector Neris,RP,1.0,740000.0,740000.0
101,Joey Wendle,3B,1.0,2000000.0,2000000.0
143,Joey Wendle,3B,1.0,740000.0,740000.0
123,Luis Guillorme,SS,1.0,1100000.0,1100000.0
144,Luis Guillorme,SS,1.0,740000.0,740000.0
98,Nick Senzel,CF,1.0,2000000.0,2000000.0
150,Nick Senzel,CF,1.0,740000.0,740000.0
106,Tim Hill,RP,1.0,1800000.0,1800000.0
149,Tim Hill,RP,1.0,740000.0,740000.0


In [6]:
df = (
    df.sort_values("total_value", ascending=False)
      .drop_duplicates(subset="player_name", keep="first")
      .sort_index()
)

print(df.shape)
print("Duplicate names:", df["player_name"].duplicated().sum())

(148, 8)
Duplicate names: 0


In [7]:
df.to_csv("../data/processed/contracts_2024_clean.csv", index=False)
print("Saved", df.shape)

Saved (148, 8)


In [8]:
import sys
sys.path.append("..")
from src.clean_contracts import clean_contracts

test = clean_contracts("../data/processed/contracts_2024.csv", 2024)
print(test.shape)
test.head()

(148, 8)


,player_name,position,years,total_value,signing_year,qualifying_offer,player_type,aav
1,Shohei Ohtani,SP,10.0,700000000.0,2024,1,pitcher,7.000000e+07
2,Yoshinobu Yamamoto,SP,12.0,325000000.0,2024,0,pitcher,2.708333e+07
3,Aaron Nola,SP,7.0,172000000.0,2024,1,pitcher,2.457143e+07
4,Jung Hoo Lee,OF,6.0,113000000.0,2024,0,hitter,1.883333e+07
5,Josh Hader,RP,5.0,95000000.0,2024,1,pitcher,1.900000e+07


In [9]:
c23 = clean_contracts("C:/Users/mvpas/Projects/fa-contract-model/data/processed/contracts_2023.csv", 2023)
print(c23.shape)
print(c23["position"].value_counts().head(8))
print("Duplicate names:", c23["player_name"].duplicated().sum())
c23.head()

(148, 8)
position
RP    41
SP    35
LF    12
1B    12
C      9
RF     9
3B     8
SS     7
Name: count, dtype: int64
Duplicate names: 0


,player_name,position,years,total_value,signing_year,qualifying_offer,player_type,aav
1,Aaron Judge,CF,9.0,360000000.0,2023,1,hitter,4.000000e+07
2,Trea Turner,SS,11.0,300000000.0,2023,1,hitter,2.727273e+07
3,Xander Bogaerts,SS,11.0,280000000.0,2023,1,hitter,2.545455e+07
4,Carlos Correa,SS,6.0,200000000.0,2023,0,hitter,3.333333e+07
5,Jacob deGrom,SP,5.0,185000000.0,2023,1,pitcher,3.700000e+07


In [10]:
raw = pd.read_csv("../data/processed/contracts_2023.csv")
print(raw.columns.tolist())
raw.head(3)

['Player (154)', 'Pos', 'Yrs', 'Value$3,900,441,666', 'AAV$13,685,760']


,Player (154),Pos,Yrs,"Value$3,900,441,666","AAV$13,685,760"
0,NaN,NaN,NaN,NaN,NaN
1,Aaron Judge QO,CF,9.0,"$360,000,000","$40,000,000"
2,Trea Turner QO,SS/CF,11.0,"$300,000,000","$27,272,727"
